# Clean-pair planning-cost adapter ablation

Select a **GPU runtime** and choose **Runtime → Run all**. This reuses the 40 saved run-0 collection episodes and their cached image tokens. It collects no new training experience and does not modify `pilot_v1`. The only training change is that a chronological pair is kept when the later saved state is at least 0.5 units closer to the goal.

The new adapter is evaluated on the same fixed 20 near and 20 far cases. Candidate-replay diagnostics are intentionally omitted.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, shutil, signal, subprocess, sys
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DRIVE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_REF = 'curriculum-cost-adapter'
SOURCE_COMMIT = None
REPO = Path('/content/i-jepa-wm-clean-pair')
DINO_ROOT = Path('/content/dino_wm')
PARENT = DRIVE / 'datasets/curriculum_cost_adapter/pilot_v1'
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/clean_pair_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE = DRIVE / 'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS = Path('/content/curriculum-clean-pair-artifacts')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'

def shell(command, *, cwd=None, log=None):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    stream = open(log, 'a', buffering=1) if log else None
    try:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    if stream: stream.write(line)
                returncode = process.wait()
            except BaseException:
                process.send_signal(signal.SIGINT)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: process.terminate()
                raise
        if returncode: raise subprocess.CalledProcessError(returncode, command)
    finally:
        if stream: stream.close()


## Set up the exact source and frozen artifacts

The child runtime records the new source revision. Parent replay files are hashed and linked through explicit parent provenance; the new code does not pretend it created the original experience.


In [ ]:
required_parent = [PARENT / 'manifest.json', PARENT / 'runtime.json', PARENT / 'report.json',
                   PARENT / 'run_0/adapter.pt', PARENT / 'run_0/training.json',
                   PARENT / 'stages/run_0/train.json']
missing = [str(path) for path in required_parent if not path.is_file()]
assert not missing, 'The completed parent pilot is missing: ' + ', '.join(missing)
assert len(list((PARENT / 'replay/run_0').glob('*.pt'))) == 40, 'Expected 40 parent replay episodes'

if not REPO.exists():
    shell(['git', 'clone', '--branch', SOURCE_REF, '--single-branch', REPO_URL, REPO])
else:
    assert (REPO / '.git').exists(), f'{REPO} is not a Git checkout'
    assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Source clone has local changes'
    shell(['git', '-C', REPO, 'fetch', 'origin', f'refs/heads/{SOURCE_REF}:refs/remotes/origin/{SOURCE_REF}'])
shell(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT or f'origin/{SOURCE_REF}'])
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Clean-pair source revision:', source_commit)

shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf', 'pyyaml', 'scipy', 'einops', 'matplotlib', 'imageio', 'imageio-ffmpeg', 'pillow'])
if not DINO_ROOT.exists(): shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO-WM source has local changes'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ENCODER, CHECKPOINT = ARTIFACTS / 'encoder.pth.tar', ARTIFACTS / 'fixed-best.pt'
for source, destination in ((ENCODER_DRIVE, ENCODER), (WM_DRIVE, CHECKPOINT)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not destination.exists() or destination.stat().st_size != source.stat().st_size: shutil.copy2(source, destination)
COMMON = ['--parent-dir', str(PARENT), '--output-dir', str(OUTPUT),
          '--world-model-checkpoint', str(CHECKPOINT), '--encoder-checkpoint', str(ENCODER),
          '--dino-wm-root', str(DINO_ROOT), '--device', 'cuda', '--precision', 'fp32', '--encoder-batch-size', '8']
def stage(name):
    OUTPUT.mkdir(parents=True, exist_ok=True)
    shell([sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_clean_pair_ablation', name, *COMMON],
          cwd=REPO, log=OUTPUT / f'{name}.log')
print('GPU:', torch.cuda.get_device_name())
print('Torch:', torch.__version__)
print('Parent:', PARENT)
print('Child output:', OUTPUT)


## Validate the parent and fit from clean pairs

`prepare` verifies the complete original run and hashes every replay file. `train` loads those cached tokens, computes and saves the frozen image-goal tokens, and runs the same 500-update schedule. There are no simulator actions in either stage.


In [ ]:
stage('prepare')
stage('train')
config = json.loads((OUTPUT / 'ablation.json').read_text())
training = json.loads((OUTPUT / 'run_0/training.json').read_text())
before = sum(row['temporal_pairs'] for row in training['trajectories'])
after = sum(row['eligible_pairs'] for row in training['trajectories'])
print(f"Replay: {config['replay_episodes']} episodes, {config['replay_frames']} frames")
print(f"Pair rule: later state must be at least {config['pair_filter']['min_goal_progress']} units closer")
print(f"Temporal pairs before oracle filter: {before}")
print(f"Clean pairs used: {after} ({after / before:.1%})")
print('Usable episodes:', len(training['training_case_ids']))
print('Saved goal tokens:', OUTPUT / 'run_0/goal_tokens.pt')


## Evaluate on the same fixed cases

This is the only stage that interacts with the simulator: 20 near and 20 far evaluations, with the original starts, goals and CEM seeds. Passing means at most two paired near losses and no aggregate far degradation relative to raw cost.


In [ ]:
stage('evaluate')
stage('report')
report = json.loads((OUTPUT / 'report.json').read_text())
print('\nCLEAN PAIRS VS RAW')
print(json.dumps(report['clean_vs_raw'], indent=2))
print('\nORIGINAL ADAPTER VS RAW')
print(json.dumps(report['original_vs_raw'], indent=2))
print('\nSCREEN:', json.dumps(report['clean_screen'], indent=2))
print('OUTCOME:', report['outcome'])
print('INTERACTION BUDGETS:', json.dumps(report['interaction_budgets'], indent=2))
from IPython.display import display, Image
for name in ('success_by_band.png', 'trajectories.png', 'training_loss.png'):
    display(Image(filename=str(OUTPUT / name)))


## Share the result

Passing the screen means clean labels prevented the previous damage; it does not establish statistical significance. More than 18/20 far successes is improvement on these fixed cases. Exactly 18/20 with retained near performance suggests label noise caused the degradation but near-only replay still lacks far coverage.

Download this executed notebook using **File → Download → Download .ipynb** and send it back for interpretation. The persisted goal tokens allow a later exact cost-ordering audit without reloading the encoder.
